# 03 — Training

Fine-tune encoder models and run LLM baselines.  
All runs use the same hyperparameters — the only variable is which density column is used for weighting.

**Sections**
1. Configuration
2. Define experiments (dataset × density column)
3. Fine-tuning runs
4. LLM baselines
5. Results summary

In [ ]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import pandas as pd

from src.training import TrainingConfig, train, run_baselines

## 1. Configuration

Edit `TrainingConfig` fields below to change model, hyperparameters, or output paths.  
All experiments in this notebook share the same config — only `density_column` varies per run.

In [ ]:
CONFIG_PATH = "configs/datasets.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg["embedding"]["output_root"]
PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
MODEL_NAME        = cfg["embedding"]["models"][0]
MODEL_SLUG        = MODEL_NAME.replace("/", "_")
K_VALUES          = cfg["embedding"]["k_values"]

# Russian is reference-only: not used for fine-tuning, only for density + eval
TRAIN_DATASETS    = cfg.get("train_datasets", ["toxigen"])

train_cfg = cfg["training"]

base_config = TrainingConfig(
    model_id     = train_cfg["models"][0],
    batch_size   = train_cfg["batch_size"],
    learning_rate= train_cfg["learning_rate"],
    num_epochs   = train_cfg["epochs"],
    max_length   = train_cfg["max_length"],
    random_state = train_cfg["random_state"],
    output_root  = train_cfg["output_root"],
)

print("Models         :", train_cfg["models"])
print("Epochs         :", base_config.num_epochs)
print("LR             :", base_config.learning_rate)
print("Train datasets :", TRAIN_DATASETS)
print("Output         :", base_config.output_root)

## 2. Define Experiments

Each experiment is `(dataset, density_column)`.  
- `density_column = None` → train without weighting (baseline encoder)
- `density_column = 'density_k5_ratio'` → weight by Russian/All ratio at K=5 on raw embeddings
- `density_column = 'density_pca_k5_ratio'` → same but PCA space

Edit the list below to add/remove experiments.

In [ ]:
# Build experiment list: one fine-tune per (train_dataset × density_column)
# Russian is excluded — it is the reference group used for density, not for training.
density_columns = [None]  # baseline: no weighting
for k in K_VALUES:
    density_columns.append(f"density_k{k}_ratio")       # raw space
    density_columns.append(f"density_pca_k{k}_ratio")   # PCA space

experiments = [
    {"dataset": ds, "density_column": dc, "model_id": mid}
    for ds in TRAIN_DATASETS
    for dc in density_columns
    for mid in train_cfg["models"]
]

print(f"{len(experiments)} fine-tunes planned:")
for ex in experiments:
    mid_slug = ex['model_id'].replace('/', '_')
    tag = f"{mid_slug}__{ex['dataset']}__{ex['density_column'] or 'no_density'}"
    print(f"  {tag}")

## 3. Fine-Tuning Runs

Each run loads `outputs/2_embeddings/{dataset}/{model_slug}/densities.csv`,  
fine-tunes the model, and saves the checkpoint + `metrics.json` to `outputs/3_training/`.

In [ ]:
all_metrics = {}

for ex in experiments:
    ds = ex["dataset"]
    dc = ex["density_column"]
    mid = ex["model_id"]
    mid_slug = mid.replace('/', '_')
    dataset_tag = ds

    density_csv = os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "densities.csv")
    if not os.path.exists(density_csv):
        print(f"[SKIP] {density_csv} not found")
        continue

    # Clone config and set density column for this run
    from dataclasses import replace
    run_config = TrainingConfig(
        model_id      = mid,
        batch_size    = base_config.batch_size,
        learning_rate = base_config.learning_rate,
        num_epochs    = base_config.num_epochs,
        max_length    = base_config.max_length,
        random_state  = base_config.random_state,
        output_root   = base_config.output_root,
        density_column= dc,
    )

    run_name = f"{mid_slug}__{run_config.run_name(dataset_tag)}"
    metrics_path = os.path.join(run_config.output_dir(dataset_tag), "metrics.json")

    if os.path.exists(metrics_path):
        print(f"[CACHE] {run_name} — loading existing metrics")
        with open(metrics_path) as f:
            all_metrics[run_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")
    metrics = train(density_csv=density_csv, dataset_tag=dataset_tag, config=run_config)
    all_metrics[run_name] = metrics

print("\nAll fine-tuning runs complete.")

## 4. LLM Baselines

Requires Ollama running locally (`ollama serve`).  
Runs zero-shot, context, and few-shot classification on the Russian test split.

In [ ]:
RUN_BASELINES = False  # set True to run (requires Ollama)

if RUN_BASELINES:
    # Use the Russian annotated test set
    russian_test_csv = os.path.join(PREPROCESSED_ROOT, "russian", "test.csv")
    if os.path.exists(russian_test_csv):
        baseline_results = run_baselines(
            test_csv=russian_test_csv,
            dataset_tag="russian_annotated",
            config=base_config,
        )
        print("Baseline results:")
        for mode, metrics in baseline_results.items():
            print(f"  {mode}: F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")
    else:
        print(f"Russian test CSV not found at {russian_test_csv}")
else:
    print("Baselines skipped (RUN_BASELINES=False).")

## 5. Results Summary

In [ ]:
if all_metrics:
    rows = []
    for run_name, m in all_metrics.items():
        parts = run_name.split("__", 2)
        rows.append({
            "run": run_name,
            "model": parts[0],
            "dataset": parts[1] if len(parts) > 1 else "n/a",
            "density": parts[2] if len(parts) > 2 else "n/a",
            "f1":               m.get("f1", float("nan")),
            "accuracy":         m.get("accuracy", float("nan")),
            "balanced_accuracy": m.get("balanced_accuracy", float("nan")),
            "auc_roc":          m.get("auc_roc", float("nan")),
        })

    summary = pd.DataFrame(rows).sort_values("f1", ascending=False)
    display(summary.style.format({
        "f1": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "auc_roc": "{:.4f}",
    }))
else:
    print("No metrics collected yet — run the training cells first.")